In [1]:
# Lab 7 
from collections import defaultdict
import time
 
state = defaultdict(set)  # simplistic, not for production
 
def process_event(event):
    user_id = event['user_id']
    ts = event['ts']  # epoch seconds
    window_key = ts - (ts % 60)  # 1-minute windows
    state[window_key].add(user_id)
 
def emit_window_counts(current_ts):
    window_key = current_ts - (current_ts % 60) - 60  # emit previous minute
    count = len(state.pop(window_key, set()))
    return {'window_start': window_key, 'unique_users': count}

In [2]:
import time

# Reset state for a clean test run (important if you re-run this cell multiple times in the same kernel session)
state.clear()

# Simulate a base timestamp aligned to a minute boundary
base_ts = 1_720_000_000  # any fixed epoch second works; doesn't need to be "now"
base_ts = base_ts - (base_ts % 60)  # align to minute boundary for predictable tests

# --- Test events within the same 1-minute window ---
test_events = [
    {'user_id': 'u1', 'ts': base_ts + 5},
    {'user_id': 'u2', 'ts': base_ts + 10},
    {'user_id': 'u1', 'ts': base_ts + 20},  # duplicate user, same window -> should NOT double count
    {'user_id': 'u3', 'ts': base_ts + 55},
]

for e in test_events:
    process_event(e)

print("State after processing:", dict(state))
assert len(state[base_ts]) == 3, f"Expected 3 unique users, got {len(state[base_ts])}"
print("✅ Dedup test passed: 3 unique users in window", base_ts)

State after processing: {1719999960: {'u2', 'u3', 'u1'}}
✅ Dedup test passed: 3 unique users in window 1719999960
